# 🪰 Drosophila Connectome Pong: TPU v5e Neuro-Arena

### Biologisch-realistische Simulation zweier virtueller Fruchtfliegen-Gehirne (*Drosophila melanogaster*)
Basierend auf dem vollständigen Konnektom des Fliegenhirns (FlyWire Consortium / Nature 2024).

### 🔬 Highlights:
- **Custom HTML5 / CSS3 / Three.js Frontend (Kein Gradio!)**: 60 FPS WebSocket-Stream.
- **Fliegen-Animationen**: Paddles sind animierte Fliegen mit dynamischem Flügelschlag (Octopamin-Frequenz).
- **3D-Konnektom-Gehirnkarten (Three.js)**: Dreidimensionale Optic Lobes, Central Complex & Mushroom Bodies.
- **Happy vs. Sauer**: Smaragdgrün & Gold bei PAM Dopamin (Treffer/Sieg); Karminrot & Orange bei PPL1 Stress/Schmerz (Miss/Niederlage).
- **RL-Skill-Level & Loss-Graphen**: Echtzeit-Berechnung des Skill-Levels (Larva $\to$ Apex Fly) und Policy Loss.
- **20-Minuten Auto-Checkpointing**: Sichert trainierte Fliegenhirne alle 20 Minuten automatisch auf Google Drive.
- **Sicherer öffentlicher Web-Host**: Automatischer, kostenloser Public Tunnel.

## 1. Hardware & TPU v5e Überprüfung

In [ ]:
# Prüfe aktive Beschleuniger (TPU v5e / GPU / CPU)
import jax
print(f"JAX Version: {jax.__version__}")
devices = jax.devices()
print(f"Erkannte Geräte ({len(devices)}): {devices}")
for i, d in enumerate(devices):
    print(f"  Gerät {i}: {d.device_kind} ({d.platform})")

## 2. Google Drive einbinden (für 20-Minuten-Dauerspeicherung)

In [ ]:
# Mountet Google Drive, damit trainierte Fliegenhirne dauerhaft erhalten bleiben
from google.colab import drive
drive.mount('/content/drive')

import os
ckpt_dir = '/content/drive/MyDrive/fly_brain_checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)
print(f"[Drive] Checkpoint-Verzeichnis bereit: {ckpt_dir}")

## 3. Abhängigkeiten installieren

In [ ]:
!pip install --upgrade -q jax[tpu] flax optax safetensors pillow fastapi uvicorn[standard] websockets

## 4. Öffentlichen Web-Host Tunnel & Simulation starten
Startet den WebSocket-Server auf Port 8000 und öffnet den sicheren öffentlichen Tunnel für den Browser.

In [ ]:
# Startet Localtunnel im Hintergrund für die öffentliche HTTPS-URL
import subprocess
subprocess.Popen(["npx", "-y", "localtunnel", "--port", "8000"])

# Startet den Drosophila TPU v5e Pong Server (60 FPS)
!python main.py --port 8000 --checkpoint-interval 20.0

## 5. Gespeicherte 20-Minuten Checkpoints einsehen

In [ ]:
import glob, json
checkpoints = glob.glob('/content/drive/MyDrive/fly_brain_checkpoints/*.json')
print(f"Gefundene Checkpoint-Metadaten ({len(checkpoints)}):")
for cp in sorted(checkpoints):
    with open(cp) as f:
        meta = json.load(f)
        print(f"- {cp}: Step {meta.get('step')}, Score: {meta.get('score1')}:{meta.get('score2')}, Fliege 1 Wins: {meta.get('fly1_total_wins')}, Fliege 2 Wins: {meta.get('fly2_total_wins')}")